## Part 4: Models, metrics and XAI (P4)

This part trains and compares the models on P3's modeling table, chooses the metrics, the hyperparameters and the decision threshold **without touching the test set**, evaluates the final choice once on the test set, explains the final model and lists its limitations.

**Inputs from Part 3:** the modeling table `model` (one row per driver per race, 2003-2024, with the `split` column), the 18 features `FEATURES` and their 6 groups `FEATURE_GROUPS`.
**Outputs for P1's inference function:** `FINAL_MODEL` (preprocessing + model in one pipeline), `FINAL_THRESHOLD` and `FINAL_NAME`.

Sections: 4.1) evaluation function `train_eval()`, 4.2) data and preprocessing, 4.3) sanity check of the metrics, 4.4) baselines, 4.5) statistical models, 4.6) shallow neural network, 4.7) hyperparameter tuning, 4.8) metric choice and threshold, 4.9) final model, 4.10) test set evaluation, 4.11) explainability (XAI), 4.12) model limitations, 4.13) hand-off to the inference function.

In [ ]:
# LOCAL SETUP (removed automatically when the notebooks are merged)
# In main.ipynb, Part 3 runs just before this part and defines `model`, FEATURES and FEATURE_GROUPS.
# Locally we get exactly the same objects by running P3's notebook first.
import sys
sys.path.append("../src")
from f1_clean import load_clean, SEED
from IPython.utils.capture import capture_output
with capture_output():          # hide Part 3's printouts and plots
    %run p3_features.ipynb

In [ ]:
clean = load_clean()
outcomes = clean["outcomes"]    # post-race columns (status, finishing position...). Never features: only used to tell the stories in 4.11-4.12
tables = clean["tables"]        # cleaned tables (driver and race names for the examples)

### 4.1 The evaluation function `train_eval()`

One function scores every model the same way, so all results (our models and P3's ablation) are directly comparable.

**Why these metrics?** The brief asks for ROC-AUC, PR-AUC and F1 for every model; we also report precision and recall, because F1 alone hides which of the two is weak.
* **ROC-AUC:** how well the model separates podium from non-podium overall (uses probabilities, no threshold).
* **PR-AUC:** how well the model ranks the real podium finishers at the top (uses probabilities). Computed as *average precision*, the standard summary of the precision-recall curve. It is more informative than ROC-AUC when the positive class is rare, and podiums are rare (about 14% of rows).
* **F1, precision, recall:** quality of the yes/no decision at a probability threshold. The threshold is 0.5 until it is tuned on validation in 4.8.

**Leakage safety:** the model is fitted on the training years only. The test set is deliberately *not* an input: it is scored once, at the very end, with `evaluate()`.

The same code is in `src/f1_eval.py`, so P3 can import it for the ablation.

In [ ]:
import pandas as pd
from sklearn.metrics import (average_precision_score, f1_score, precision_score,
                             recall_score, roc_auc_score)


def evaluate(model, X, y, threshold=0.5):
    """Score an already-fitted model on one split and return the 5 metrics.
    ROC-AUC and PR-AUC use the probabilities (no threshold needed);
    F1, precision and recall use the yes/no prediction at `threshold`."""
    proba = model.predict_proba(X)[:, 1]          # predicted probability of a podium
    pred = (proba >= threshold).astype(int)       # 1 = "podium" at this threshold
    return {
        "ROC-AUC": roc_auc_score(y, proba),
        "PR-AUC": average_precision_score(y, proba),
        "F1": f1_score(y, pred, zero_division=0),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
    }


def train_eval(model, X_train, y_train, X_val, y_val, threshold=0.5, name="model"):
    """Fit `model` on the training years only, then score it on train and validation.
    The test set is not used here on purpose: it is scored once, at the end, with evaluate()."""
    model.fit(X_train, y_train)
    rows = [
        {"model": name, "split": "train", **evaluate(model, X_train, y_train, threshold)},
        {"model": name, "split": "validation", **evaluate(model, X_val, y_val, threshold)},
    ]
    return pd.DataFrame(rows).round(4)

### 4.2 Data and preprocessing

**Data:** P3's modeling table and P3's time split (train 2003-2019, validation 2020-2021, test 2022-2024). All models use the same 18 features. The test rows are separated only in 4.10.

**Preprocessing (one pipeline, the same for every model):**
* **Missing values are structural, not errors:** standings are empty in round 1 of every season (no race before it), form is empty for a driver's first race, and qualifying times are missing for about 1-2% of rows. P3 left them as `NaN` on purpose. We fill them with the **training median** and add a **missing-indicator column** for every feature that has gaps, so the model still knows that the value was missing (for example "first race of the season").
* **Scaling:** every numeric feature is standardized (mean 0, standard deviation 1). Logistic regression and the network need it; for the tree models it changes nothing (trees only compare values within one feature), so all models can share one pipeline.
* **`era`** (`v8`, `hybrid`, `ground_effect`) is a category, so it is one-hot encoded (one 0/1 column per era). **Caveat:** `ground_effect` exists only in the test years (2022+). The encoder ignores unseen categories, so a test row gets 0 in every era column, i.e. "an era the model has not seen". The validation years are all `hybrid`, so this feature cannot be checked on validation; see 4.12.
* **Fitted on the training rows only:** the imputer, scaler and encoder are steps of a `Pipeline`, so `fit()` learns the medians, means and categories from the training years only, and the validation and test rows are only transformed.

In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

table = model                     # P3's modeling table (read only; the name `model` is kept for Part 3)
assert not set(FEATURES) & set(outcomes.columns), "a post-race column is used as a feature"

train = table[table["split"] == "train"]
val = table[table["split"] == "validation"]
X_train, y_train = train[FEATURES], train["podium"]
X_val, y_val = val[FEATURES], val["podium"]

CATEGORICAL = [f for f in FEATURES if not pd.api.types.is_numeric_dtype(table[f])]   # era
NUMERIC = [f for f in FEATURES if f not in CATEGORICAL]
preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
])


def with_preprocessing(estimator):
    """The shared preprocessing followed by `estimator`, as one pipeline (fitted on the training rows only)."""
    return Pipeline([("prep", clone(preprocess)), ("model", estimator)])


for name, part in [("train", train), ("validation", val)]:
    print(f"{name}: {len(part)} rows ({part['year'].min()}-{part['year'].max()}), "
          f"{part['podium'].sum()} podiums, podium rate {part['podium'].mean():.3f}")
display(X_train.isna().mean().loc[lambda s: s > 0].round(3).to_frame("share missing (train)"))

### 4.3 Sanity check of the metrics

Before trusting the function, we test it on a model that knows nothing: `DummyClassifier` always predicts the training podium rate. A correct implementation must give **ROC-AUC = 0.5** (no separation), **PR-AUC = the podium rate** (no better than random ranking) and **F1 = 0** at threshold 0.5 (it never says "podium"). We also print its accuracy, to show why accuracy is not one of our metrics.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy="prior", random_state=SEED)
dummy_results = train_eval(dummy, X_train, y_train, X_val, y_val, name="dummy (always the base rate)")
display(dummy_results)

val_row = dummy_results[dummy_results["split"] == "validation"].iloc[0]
assert abs(val_row["ROC-AUC"] - 0.5) < 1e-3
assert abs(val_row["PR-AUC"] - y_val.mean()) < 1e-3
assert val_row["F1"] == 0
print(f"Dummy accuracy on validation: {accuracy_score(y_val, dummy.predict(X_val)):.1%} "
      f"(looks good, but it finds 0 podiums)")

**What we see.** The dummy model gets ROC-AUC 0.5, PR-AUC equal to the podium rate (0.151 on validation) and F1 = 0, exactly as expected, so the function computes the metrics correctly. Its accuracy is still 84.9%, although it never finds a single podium. That is why accuracy is not one of our metrics, and why **PR-AUC is the natural primary metric** for this rare positive class. These numbers are the floor every real model must beat.

### 4.4 Baselines

A baseline is the simplest reasonable approach. Every later model must beat it, otherwise its extra complexity is not earned.
* **Rule `grid <= 3`:** predict a podium for the three cars that start at the front. It learns nothing; it encodes the domain fact that the starting position is the strongest single predictor.
* **Logistic regression:** the simplest statistical model. It multiplies each feature by a learned weight, adds them up, and turns the sum into a probability with the sigmoid function. *Limitation:* it is linear, so it cannot learn interactions on its own (for example "a good grid slot matters more for a top team").

#### Attempt 1: rule `grid <= 3`
The rule gives a hard yes/no, so its ROC-AUC and PR-AUC are those of a single operating point. `grid = 0` (pit-lane start) is not a front-row start.

In [ ]:
class GridRule:
    """Rule baseline: 'podium' if the car starts in grid slot 1, 2 or 3. Learns nothing."""
    def fit(self, X, y):
        return self

    def predict_proba(self, X):
        p = X["grid"].between(1, 3).astype(float).to_numpy()
        return np.column_stack([1 - p, p])

rule_results = train_eval(GridRule(), X_train, y_train, X_val, y_val, name="rule grid <= 3")
display(rule_results)

#### Attempt 2: logistic regression (all 18 features)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = with_preprocessing(LogisticRegression(max_iter=2000, random_state=SEED))
lr_results = train_eval(lr, X_train, y_train, X_val, y_val, name="logistic regression")
display(lr_results)

#### Attempt 3: logistic regression with `class_weight="balanced"`
Podiums are only about 14% of the training rows. `class_weight="balanced"` gives the podium rows a larger weight, so both classes count equally in training. We expect the ranking metrics (ROC-AUC, PR-AUC) to stay about the same and the yes/no metrics at threshold 0.5 to change, because the weighting mostly moves the predicted probabilities up.

In [ ]:
lr_bal = with_preprocessing(LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
lr_bal_results = train_eval(lr_bal, X_train, y_train, X_val, y_val, name="logistic regression (balanced)")
display(lr_bal_results)

#### Attempt 4: logistic regression on the 4 draft features only
Before P3's table was ready, the models were built on four raw pre-race columns (`grid`, `pitlane_start`, `has_qualifying`, `qual_position`). Same model, same rows: does the feature engineering add anything?

In [ ]:
DRAFT_FEATURES = ["grid", "pitlane_start", "has_qualifying", "qual_position"]
lr_draft = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(max_iter=2000, random_state=SEED))
lr_draft_results = train_eval(lr_draft, train[DRAFT_FEATURES].astype(float), y_train,
                              val[DRAFT_FEATURES].astype(float), y_val, name="logistic regression (4 draft features)")
display(lr_draft_results)

#### Comparison of the baselines (validation)

In [ ]:
import matplotlib.pyplot as plt

baselines = pd.concat([dummy_results, rule_results, lr_results, lr_bal_results, lr_draft_results], ignore_index=True)
val_table = baselines[baselines["split"] == "validation"].set_index("model").drop(columns="split")
display(val_table)

ax = val_table[["ROC-AUC", "PR-AUC", "F1"]].plot.barh(figsize=(8, 4))
ax.set_title("Baselines on validation (2020-2021), threshold 0.5")
ax.set_xlabel("score (higher is better)")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(title="metric", loc="lower right")
plt.tight_layout()
plt.show()

# what the logistic regression learned: weights of the standardized inputs, largest first
weights = pd.Series(lr.named_steps["model"].coef_[0], index=lr.named_steps["prep"].get_feature_names_out())
display(weights.sort_values(key=abs, ascending=False).head(10).round(3).to_frame("weight (standardized input)"))

**What we see (validation 2020-2021).**
* **Logistic regression is far above both references:** PR-AUC 0.738 vs 0.530 for the rule and 0.151 for the dummy; ROC-AUC 0.933 vs 0.820.
* **P3's features add real information:** the same logistic regression on the same rows reaches PR-AUC 0.672 with the 4 draft features and **0.738 with the 18 features (+0.066)**. This is the gain of the feature engineering (standings, form and the qualifying details); which group brings how much is the subject of P3's ablation and of the XAI in 4.11.
* **At threshold 0.5 the rule still has the best F1** (0.695 vs 0.679): plain logistic regression is cautious at 0.5 (precision 0.73, recall 0.63). The threshold matters as much as the model, so it is tuned in 4.8.
* **`class_weight="balanced"` behaves as expected:** PR-AUC barely changes (0.733 vs 0.738) because the ranking stays the same, while at 0.5 recall rises to 0.85 and precision falls to 0.57. Weighting mostly moves the probabilities, which threshold tuning can also do.
* **Weights:** the largest are `qual_position` (-1.24) and `grid` (-0.89): a position nearer the front means a higher podium probability; then the championship position before the race, `pos_before` (-0.45). Several features carry the same information (`grid` and `qual_position`; `reached_q2`, `reached_q3` and `qual_position`; points and positions), so the sign of a single weight should not be over-read. These are associations the model uses, not causes.
* **Train PR-AUC is a little lower than validation** (0.690 vs 0.738). The training years 2003-2019 include seasons with less predictable results; with P3's start year (2003) this gap is much smaller than with the draft data that started in 1950 (about -0.14).

**Baseline to beat:** logistic regression, validation PR-AUC **0.738**.

### 4.5 Statistical models: random forest and gradient boosting

Two tree-based models, chosen because they can learn what logistic regression cannot: non-linear effects and interactions between features (for example "a good grid slot matters more for a team in good form"). Both are in scikit-learn, so no extra library is needed.
* **Random forest:** many decision trees, each trained on a random sample of rows and features; the prediction is the average vote. Averaging many different trees reduces overfitting. *Limitation:* its probabilities are averages of votes (not calibrated), and it cannot extrapolate beyond the values seen in training.
* **Gradient boosting** (`HistGradientBoostingClassifier`): small trees built one after another, each one correcting the errors of the trees before it. Usually the strongest model family on tables like ours. *Limitation:* more hyperparameters, and it can overfit if it runs too long (scikit-learn stops early on a 10% slice of the training rows, never on our validation years).

**Class imbalance:** both use `class_weight="balanced"` (podiums are about 14% of rows). Section 4.4 showed this mostly moves the probabilities; the threshold is tuned in 4.8. Hyperparameters are tuned in 4.7.

#### Attempt 5: random forest
300 trees; `min_samples_leaf=20` so every leaf is based on at least 20 races, which keeps the probabilities smooth instead of memorizing single rows.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = with_preprocessing(RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced",
                                               n_jobs=-1, random_state=SEED))
rf_results = train_eval(rf, X_train, y_train, X_val, y_val, name="random forest")
display(rf_results)

#### Attempt 6: gradient boosting
Default settings for now.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = with_preprocessing(HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED))
gb_results = train_eval(gb, X_train, y_train, X_val, y_val, name="gradient boosting")
display(gb_results)

#### Comparison of all models so far
Validation scores for every attempt, plus the train-validation gap in PR-AUC (a large positive gap means overfitting).

In [ ]:
def compare(results):
    """Validation scores of every model, plus the train - validation PR-AUC gap."""
    allres = pd.concat(results, ignore_index=True)
    pr = allres.pivot(index="model", columns="split", values="PR-AUC")
    summary = allres[allres["split"] == "validation"].set_index("model").drop(columns="split")
    summary["PR-AUC gap (train - val)"] = (pr["train"] - pr["validation"]).round(4)
    return summary.sort_values("PR-AUC", ascending=False)

summary = compare([dummy_results, rule_results, lr_results, lr_bal_results, lr_draft_results, rf_results, gb_results])
display(summary)

lr_pr = summary.loc["logistic regression", "PR-AUC"]
ax = summary["PR-AUC"].sort_values().plot.barh(figsize=(8, 4), color="tab:orange")
ax.axvline(lr_pr, color="black", linestyle="--", label=f"baseline to beat (logistic regression) = {lr_pr:.4f}")
ax.set_title("PR-AUC on validation (2020-2021), all models so far")
ax.set_xlabel("PR-AUC (higher is better)")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**What we see (validation 2020-2021).**
* **The tree models do not beat logistic regression:** PR-AUC random forest 0.726, gradient boosting 0.704, logistic regression 0.738; ROC-AUC is 0.926-0.933 for all three.
* **Gradient boosting with default settings overfits:** train PR-AUC 0.932 vs validation 0.704 (gap +0.23). With about 960 training podiums, trees with 31 leaves learn details of the training seasons that do not repeat. The random forest (`min_samples_leaf=20`) overfits much less (gap +0.045). Tuning (4.7) controls the tree size.
* **Why no gain from trees:** the strongest signal is a smooth "the nearer the front, the likelier the podium" relation that a straight line already captures; the extra interactions the trees can learn are small compared with the noise of 117 validation podiums.
* **At threshold 0.5** both balanced tree models behave like the balanced logistic regression: high recall (0.77-0.88), lower precision (0.53-0.59).

**Decision:** keep both as our two statistical models and tune them in 4.7. The baseline to beat is still logistic regression.

### 4.6 Shallow neural network (PyTorch)

**How it works:** a feed-forward network. Each hidden layer computes weighted sums of its inputs and passes them through a ReLU (negative values become 0), which lets the network learn non-linear patterns. The output layer gives one score that the sigmoid turns into a podium probability. The weights are learned by gradient descent (Adam) on the binary cross-entropy loss. With no hidden layer this would be exactly logistic regression, so the hidden layers are what the network adds.

**Choices** ("shallow" = 1-2 hidden layers, as the brief asks):
* **Dropout 0.2:** during training, 20% of the hidden units are switched off at random in every batch, so the network cannot rely on single units (less overfitting).
* **Early stopping on validation:** after every epoch (one pass over the training rows) we compute the validation loss; training stops when it has not improved for 15 epochs, and the weights of the best epoch are kept.
* **Class imbalance:** `pos_weight` = (non-podium rows / podium rows) in the loss, the same idea as `class_weight="balanced"`.
* **Preprocessing:** the same pipeline as the other models (4.2), fitted on the training rows only inside `fit()`.
* **Reproducible:** `torch.manual_seed(SEED)`, a seeded batch order and a single CPU thread (the number of threads changes the rounding of the sums), so Run All gives the same numbers on a laptop and on Kaggle.

*Limitations:* many settings (layers, units, dropout, learning rate), needs scaling and imputation, a black box without XAI, and on small tables it rarely beats tree models. **Caveat:** the validation set chooses the stopping epoch, so the network's validation score is slightly optimistic compared with the other models; the test set gives the fair comparison.

The class below only wraps the network with `fit` / `predict_proba`, so the same `train_eval()` scores it.

In [ ]:
import torch
from torch import nn


class ShallowNN:
    """Feed-forward network with 1-2 hidden layers, wrapped with fit / predict_proba so train_eval() can score it.
    - preprocessing: the shared pipeline of 4.2, fitted on the training rows only, inside fit()
    - early stopping: training stops when the validation loss has not improved for `patience` epochs,
      and the weights of the best epoch are kept"""

    def __init__(self, X_val, y_val, hidden=(32,), dropout=0.2, lr=1e-3, batch_size=256,
                 max_epochs=200, patience=15, seed=SEED):
        self.X_val, self.y_val = X_val, y_val
        self.hidden, self.dropout, self.lr, self.batch_size = hidden, dropout, lr, batch_size
        self.max_epochs, self.patience, self.seed = max_epochs, patience, seed

    def _build(self, n_inputs):
        layers, size = [], n_inputs
        for h in self.hidden:                                   # hidden layer: linear -> ReLU -> dropout
            layers += [nn.Linear(size, h), nn.ReLU(), nn.Dropout(self.dropout)]
            size = h
        layers.append(nn.Linear(size, 1))                       # output: one score (logit) for "podium"
        return nn.Sequential(*layers)

    def _tensor(self, X):
        return torch.tensor(self.prep.transform(X), dtype=torch.float32)

    def fit(self, X, y):
        torch.manual_seed(self.seed)
        torch.set_num_threads(1)                                # same numbers on every machine
        self.prep = clone(preprocess).fit(X)
        Xt, yt = self._tensor(X), torch.tensor(np.asarray(y), dtype=torch.float32)
        Xv, yv = self._tensor(self.X_val), torch.tensor(np.asarray(self.y_val), dtype=torch.float32)

        self.net = self._build(Xt.shape[1])
        pos_weight = (yt == 0).sum() / (yt == 1).sum()          # class imbalance, like class_weight="balanced"
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(self.net.parameters(), lr=self.lr)
        gen = torch.Generator().manual_seed(self.seed)

        self.history = {"train": [], "validation": []}
        best_loss, best_state, waited = float("inf"), None, 0
        for epoch in range(self.max_epochs):
            self.net.train()                                   # dropout on
            batch_losses = []
            for idx in torch.randperm(len(Xt), generator=gen).split(self.batch_size):
                optimizer.zero_grad()
                loss = loss_fn(self.net(Xt[idx]).squeeze(1), yt[idx])
                loss.backward()
                optimizer.step()
                batch_losses.append(loss.item())
            self.net.eval()                                    # dropout off
            with torch.no_grad():
                val_loss = loss_fn(self.net(Xv).squeeze(1), yv).item()
            self.history["train"].append(float(np.mean(batch_losses)))
            self.history["validation"].append(val_loss)

            if val_loss < best_loss:                           # early stopping on the validation loss
                best_loss, waited = val_loss, 0
                best_state = {k: v.clone() for k, v in self.net.state_dict().items()}
                self.best_epoch = epoch + 1
            else:
                waited += 1
                if waited >= self.patience:
                    break
        self.net.load_state_dict(best_state)                   # keep the best epoch, not the last one
        return self

    def predict_proba(self, X):
        self.net.eval()
        with torch.no_grad():
            p = torch.sigmoid(self.net(self._tensor(X)).squeeze(1)).numpy()
        return np.column_stack([1 - p, p])

#### Attempt 7: one hidden layer (32 units)

In [ ]:
nn1 = ShallowNN(X_val, y_val, hidden=(32,))
nn1_results = train_eval(nn1, X_train, y_train, X_val, y_val, name="neural network (32)")
display(nn1_results)
print(f"stopped after {len(nn1.history['train'])} epochs, best epoch: {nn1.best_epoch}")

#### Attempt 8: two hidden layers (64 and 32 units)

In [ ]:
nn2 = ShallowNN(X_val, y_val, hidden=(64, 32))
nn2_results = train_eval(nn2, X_train, y_train, X_val, y_val, name="neural network (64-32)")
display(nn2_results)
print(f"stopped after {len(nn2.history['train'])} epochs, best epoch: {nn2.best_epoch}")

#### Loss curves
Training loss (average over the batches of each epoch, with dropout on) and validation loss per epoch. The dashed line marks the epoch whose weights were kept.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, net, title in [(axes[0], nn1, "1 hidden layer (32)"), (axes[1], nn2, "2 hidden layers (64-32)")]:
    epochs = range(1, len(net.history["train"]) + 1)
    ax.plot(epochs, net.history["train"], label="training loss")
    ax.plot(epochs, net.history["validation"], label="validation loss")
    ax.axvline(net.best_epoch, color="black", linestyle="--", label=f"best epoch ({net.best_epoch})")
    ax.set_title(f"Neural network, {title}")
    ax.set_xlabel("epoch")
    ax.legend()
axes[0].set_ylabel("weighted binary cross-entropy")
plt.tight_layout()
plt.show()

#### Comparison of all models so far (validation)

In [ ]:
display(compare([dummy_results, rule_results, lr_results, lr_bal_results, lr_draft_results,
                 rf_results, gb_results, nn1_results, nn2_results]))

**What we see (validation 2020-2021).**
* **The two-layer network has the highest validation PR-AUC so far:** 0.752, vs 0.738 for logistic regression (+0.014); the one-layer network gets 0.735. We do **not** call this a win yet: the validation set chose the stopping epoch (so the network's score is slightly optimistic), and with 117 validation podiums a difference this small is within noise (measured in 4.9).
* **Loss curves:** both losses fall quickly in the first 5-10 epochs and then flatten. The validation loss never turns clearly upward, and early stopping keeps the best epoch (84 for one layer, 24 for two layers), so dropout and early stopping prevent overfitting. The validation loss stays a little above the training loss, a normal small gap, and the train-validation PR-AUC gap is small (-0.03 to -0.05, like logistic regression).
* **At threshold 0.5** the networks behave like the other balanced models: high recall (0.86-0.87), low precision (0.50-0.51).

### 4.7 Hyperparameter tuning (validation only)

For each model we try a small grid of the settings that control its complexity. Every combination is trained on the training years and scored on validation with `train_eval()`; the best combination is the one with the highest **validation PR-AUC** (our primary metric, see 4.8). The test years are not touched.

The grids are deliberately small: the validation set has only about 117 podiums, so a large search would mostly fit its noise.

In [ ]:
from sklearn.model_selection import ParameterGrid

def tune(name, make_model, grid):
    """Train one model per combination in `grid` (training years) and score it on validation.
    Returns the fitted model with the best validation PR-AUC and the table of all results."""
    rows, best_model, best_pr = [], None, -1
    for params in ParameterGrid(grid):
        candidate = make_model(**params)
        res = train_eval(candidate, X_train, y_train, X_val, y_val, name=name).set_index("split")
        rows.append({**params, "train PR-AUC": res.loc["train", "PR-AUC"],
                     "val PR-AUC": res.loc["validation", "PR-AUC"], "val ROC-AUC": res.loc["validation", "ROC-AUC"]})
        if res.loc["validation", "PR-AUC"] > best_pr:
            best_model, best_pr = candidate, res.loc["validation", "PR-AUC"]
    return best_model, pd.DataFrame(rows).sort_values("val PR-AUC", ascending=False, ignore_index=True)

tuned = {}   # model name -> (best fitted model, table of all combinations)

#### Attempt 9: tuning logistic regression
`C` is the inverse regularization strength: a small `C` pushes the weights towards 0 (simpler model), a large `C` lets them grow.

In [ ]:
tuned["logistic regression"] = tune(
    "logistic regression",
    lambda C: with_preprocessing(LogisticRegression(C=C, max_iter=2000, random_state=SEED)),
    {"C": [0.01, 0.1, 1, 10]})
display(tuned["logistic regression"][1])

#### Attempt 10: tuning the random forest
`min_samples_leaf`: the minimum number of races behind each leaf (larger = smoother, less memorizing). `max_features`: how many inputs each split may look at (`"sqrt"` = a random subset, `1.0` = all of them; a subset makes the trees more different from each other).

In [ ]:
tuned["random forest"] = tune(
    "random forest",
    lambda min_samples_leaf, max_features: with_preprocessing(
        RandomForestClassifier(n_estimators=300, min_samples_leaf=min_samples_leaf, max_features=max_features,
                               class_weight="balanced", n_jobs=-1, random_state=SEED)),
    {"min_samples_leaf": [5, 20, 50], "max_features": ["sqrt", 1.0]})
display(tuned["random forest"][1])

#### Attempt 11: tuning gradient boosting
`learning_rate`: how much each new tree may correct the previous ones (smaller = slower, more careful). `max_leaf_nodes`: the size of each tree (smaller = simpler trees).

In [ ]:
tuned["gradient boosting"] = tune(
    "gradient boosting",
    lambda learning_rate, max_leaf_nodes: with_preprocessing(HistGradientBoostingClassifier(
        learning_rate=learning_rate, max_leaf_nodes=max_leaf_nodes, class_weight="balanced", random_state=SEED)),
    {"learning_rate": [0.03, 0.1], "max_leaf_nodes": [7, 15, 31]})
display(tuned["gradient boosting"][1])

#### Attempt 12: tuning the neural network
`hidden`: the size of the hidden layer(s). `dropout`: the share of hidden units switched off during training. (The number of epochs is already chosen by early stopping.)

In [ ]:
tuned["neural network"] = tune(
    "neural network",
    lambda hidden, dropout: ShallowNN(X_val, y_val, hidden=hidden, dropout=dropout),
    {"hidden": [(16,), (32,), (64, 32)], "dropout": [0.2, 0.4]})
display(tuned["neural network"][1])

**What we see (tuning, validation PR-AUC).**
* **Logistic regression:** best with `C=10` (0.738); strong regularization (`C=0.01`) costs 0.02 (0.718). With 18 features and about 6,800 rows, little regularization is needed.
* **Random forest:** best with `max_features="sqrt"` and `min_samples_leaf=5`: 0.729 (0.726 before), but its train PR-AUC is 0.908, a large gap: small leaves memorize the training seasons.
* **Gradient boosting:** best with the **smallest and slowest** setting (`learning_rate=0.03`, `max_leaf_nodes=7`): 0.729 (0.704 with the defaults), and the overfitting of 4.5 disappears (train 0.712). The six settings are ordered almost exactly by tree size: the bigger the trees, the worse the validation score.
* **Neural network:** best with two layers (64-32) and dropout 0.2: 0.752, the setting already used in 4.6.
* **Tuning gains are small** (at most +0.025, for gradient boosting), and the four tuned models end within 0.025 of each other, less than the noise of the validation PR-AUC (standard error 0.040, see 4.9).

### 4.8 Metric choice and decision threshold

**Primary metric: PR-AUC.** Podiums are rare (about 15% of the validation rows). PR-AUC measures how well the real podium finishers are ranked at the top, it does not depend on a threshold, and its floor is the podium rate (the dummy in 4.3) instead of 0.5, so it shows clearly how much a model adds.

**Secondary metric: F1 at the chosen threshold.** In the end the model must say yes or no; F1 balances precision (how many predicted podiums are right) and recall (how many real podiums are found).

ROC-AUC is reported for completeness, but with about 85% non-podium rows it looks good easily (about 0.8 even for the simple rule).

**Threshold:** for every tuned model we try thresholds from 0.05 to 0.95 (step 0.01) on validation and keep the one with the highest F1. Each model gets its own threshold, because class weighting shifts the probabilities (4.4).

In [ ]:
def best_threshold(clf, X, y):
    """Threshold between 0.05 and 0.95 (step 0.01) with the highest F1 on (X, y)."""
    proba = clf.predict_proba(X)[:, 1]
    candidates = np.round(np.arange(0.05, 0.96, 0.01), 2)
    f1s = [f1_score(y, proba >= t, zero_division=0) for t in candidates]
    return candidates[int(np.argmax(f1s))]

rows = []
for name, (clf, _) in tuned.items():
    t = best_threshold(clf, X_val, y_val)
    rows.append({"model": name, "threshold": t, **evaluate(clf, X_val, y_val, threshold=t)})
tuned_table = pd.DataFrame(rows).set_index("model").round(4)
thresholds = tuned_table["threshold"].to_dict()
display(tuned_table)

### 4.9 Choosing the final model (validation only)

Before comparing the models we measure how noisy the validation PR-AUC is: we resample the validation rows with replacement 1000 times (bootstrap) and take the standard deviation of PR-AUC as its standard error (SE).

**Rule (one-standard-error rule):** among the models whose validation PR-AUC is within one SE of the best, take the simplest (order: logistic regression < random forest < gradient boosting < neural network). A difference smaller than the noise is not a reason to accept a more complex model. The final model and its threshold are then used once on the test set (4.10) and by P1's inference function (4.13).

In [ ]:
COMPLEXITY = ["logistic regression", "random forest", "gradient boosting", "neural network"]   # simplest first
val_pr = tuned_table["PR-AUC"]
best_name = val_pr.idxmax()

# bootstrap standard error of the best model's validation PR-AUC
rng = np.random.default_rng(SEED)
proba, y = tuned[best_name][0].predict_proba(X_val)[:, 1], y_val.to_numpy()
boot = []
for _ in range(1000):
    i = rng.integers(0, len(y), len(y))
    boot.append(average_precision_score(y[i], proba[i]))
se = np.std(boot)

within_one_se = [name for name in COMPLEXITY if val_pr[name] >= val_pr[best_name] - se]
FINAL_NAME = within_one_se[0]
FINAL_MODEL, FINAL_THRESHOLD = tuned[FINAL_NAME][0], thresholds[FINAL_NAME]

print(f"best validation PR-AUC: {best_name} = {val_pr[best_name]:.4f}, bootstrap SE = {se:.4f} "
      f"(95% range {np.percentile(boot, 2.5):.3f}-{np.percentile(boot, 97.5):.3f})")
print(f"models within one SE: {within_one_se}")
print(f"final model: {FINAL_NAME}, threshold {FINAL_THRESHOLD}")

**How the final threshold was chosen:** precision, recall and F1 of the final model on validation for every threshold.

In [ ]:
proba = FINAL_MODEL.predict_proba(X_val)[:, 1]
candidates = np.round(np.arange(0.05, 0.96, 0.01), 2)
curve = pd.DataFrame([evaluate(FINAL_MODEL, X_val, y_val, threshold=t) for t in candidates], index=candidates)

ax = curve[["precision", "recall", "F1"]].plot(figsize=(8, 4))
ax.axvline(FINAL_THRESHOLD, color="black", linestyle="--", label=f"chosen threshold = {FINAL_THRESHOLD}")
ax.set_title(f"Final model ({FINAL_NAME}): metrics vs threshold on validation")
ax.set_xlabel("threshold (predict podium if probability >= threshold)")
ax.set_ylabel("score")
ax.set_ylim(0, 1)
ax.legend(loc="center right")
plt.tight_layout()
plt.show()

# compare the final yes/no decisions with the simple rule grid <= 3 from 4.4
rule_pred = X_val["grid"].between(1, 3).to_numpy()
disagree = int(((proba >= FINAL_THRESHOLD) != rule_pred).sum())
print(f"rows where the final model and the rule grid <= 3 disagree: {disagree} of {len(X_val)}")

**What we see (threshold and final model).**
* **Each model needs its own threshold:** 0.23 for logistic regression, 0.80 for the random forest, 0.74 for gradient boosting, 0.81 for the network. The balanced models have their probabilities pushed up by the class weights, so their best threshold is higher; plain logistic regression is cautious, so its best threshold is below 0.5.
* **Tuning the threshold raised F1 to about 0.71 for every model** (logistic regression 0.717, network 0.717, gradient boosting 0.713, random forest 0.706), from 0.63-0.68 at threshold 0.5.
* **Final model:** the bootstrap standard error of the validation PR-AUC is 0.040 (95% range 0.67-0.83). All four models are within one standard error of the best (the network, 0.752), so the one-standard-error rule picks the simplest: **logistic regression with threshold 0.23**.
* **Threshold plot:** on validation F1 is almost flat between about 0.23 and 0.47 (0.69-0.72); 0.23 is the highest point but sits at the low end of that plateau, where recall is high and precision lower. A small shift of the probabilities can therefore move many rows across the threshold (this matters on the test set, 4.10 and 4.12).
* **More than the rule:** the final model disagrees with the rule `grid <= 3` on 41 of 774 validation rows, so with P3's features it no longer just repeats "the front three on the grid" (with the 4 draft features it disagreed on only 2 rows).

### 4.10 Final evaluation on the test set (used once)

Until now the test years (2022-2024) were never used: every model, hyperparameter, threshold and the final choice were fixed on the training and validation years. Here each tuned model is scored **once** on the test set with the threshold it got on validation, next to its train and validation scores, so we can see whether the validation results hold on unseen seasons. Nothing is changed after this point. The rule `grid <= 3` is included as the reference to beat.

In [ ]:
test = table[table["split"] == "test"]
X_test, y_test = test[FEATURES], test["podium"]
print(f"test: {len(X_test)} rows ({test['year'].min()}-{test['year'].max()}), {y_test.sum()} podiums, "
      f"podium rate {y_test.mean():.3f} (= the PR-AUC of a model that knows nothing)")

final_models = {"rule grid <= 3": (GridRule(), 0.5),
                **{name: (clf, thresholds[name]) for name, (clf, _) in tuned.items()}}
rows = []
for name, (clf, t) in final_models.items():
    label = f"{name} (final)" if name == FINAL_NAME else name
    for split, X, y in [("train", X_train, y_train), ("validation", X_val, y_val), ("test", X_test, y_test)]:
        rows.append({"model": label, "split": split, "threshold": t, **evaluate(clf, X, y, threshold=t)})
comparison = pd.DataFrame(rows).round(4)
display(comparison)

# the two metrics we chose (4.8), side by side per split
display(comparison.pivot(index="model", columns="split", values=["PR-AUC", "F1"])
        .reindex(columns=["train", "validation", "test"], level=1))

#### ROC and precision-recall curves (test set)
The ROC curve shows the true-positive rate against the false-positive rate for every threshold; the precision-recall curve shows precision against recall. The area under each curve is the ROC-AUC and the PR-AUC (`AP`) in the table.

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13, 5))
for name, (clf, _) in tuned.items():
    proba = clf.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=ax_roc)
    PrecisionRecallDisplay.from_predictions(y_test, proba, name=name, ax=ax_pr)
ax_roc.plot([0, 1], [0, 1], color="black", linestyle="--", label="random guess (AUC = 0.5)")
ax_pr.axhline(y_test.mean(), color="black", linestyle="--", label=f"podium rate = {y_test.mean():.3f}")
ax_roc.set_title("ROC curves on the test set (2022-2024)")
ax_pr.set_title("Precision-recall curves on the test set (2022-2024)")
ax_roc.legend(loc="lower right")
ax_pr.legend(loc="lower left")
plt.tight_layout()
plt.show()

#### Confusion matrices (test set, threshold chosen on validation)
Rows are the real outcome, columns the prediction. Top-right = false alarms (predicted a podium that did not happen), bottom-left = missed podiums.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, (clf, _)) in zip(axes, tuned.items()):
    pred = (clf.predict_proba(X_test)[:, 1] >= thresholds[name]).astype(int)
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["no podium", "podium"],
                                            ax=ax, colorbar=False)
    ax.set_title(f"{name}{' (final)' if name == FINAL_NAME else ''}\nthreshold {thresholds[name]}")
plt.tight_layout()
plt.show()

**What we see (test set 2022-2024, 1,351 rows, 204 podiums).**
* **Ranking (PR-AUC, our primary metric): the final model is the best of all models on unseen seasons:** logistic regression 0.712, network 0.698, gradient boosting 0.690, random forest 0.678; rule 0.474; no-skill floor 0.151. ROC-AUC is 0.93-0.94 for all four. The drop from validation is at most about 0.05, about one standard error (0.040). The one-standard-error rule paid off: the network, best on validation (0.752), falls to 0.698 (its stopping epoch was chosen on validation), while the simplest model holds best.
* **The random forest overfitted:** train PR-AUC 0.908, test 0.678, the largest drop of all models.
* **Yes/no decisions (F1): the final model is the weakest on the test set.** F1 0.581 (precision 0.42, recall 0.95): it finds 194 of the 204 podiums but raises 270 false alarms. Gradient boosting has the best test F1 (0.689), then the network (0.672) and the rule (0.649).
* **Why:** the final model's probabilities are too high on the test years (4.12: about 5 predicted podiums per race instead of 3). The cause is `era`: the test seasons are the `ground_effect` era, which never occurs in training. The encoder gives such rows 0 in every era column, and logistic regression learned negative weights for both known eras, so every test row gets the same push upwards. A push that is the same for every row does not change the ranking (the PR-AUC stays the best), but it moves many rows over the low threshold 0.23. We flagged this risk in 4.2 before any result.
* **We do not change anything after seeing these results:** choosing a new threshold, feature set or model on the test set would turn the test set into a second validation set, and its score would no longer be an honest estimate. The fix is a recommendation in 4.12.

### 4.11 Explainability (XAI) of the final model

We explain the final model chosen in 4.9 (`FINAL_MODEL`). The code only calls `predict_proba`, so it works for any of our models. We use the validation years: explanations do not choose anything, so there is no reason to touch the test set again.

* **Permutation importance (global):** shuffle one feature at a time and measure how much the validation PR-AUC drops. A big drop means the model relies on that feature. Because several features carry the same information (for example `grid` and `qual_position`), we also shuffle each of P3's **feature groups** together: shuffling only one of two twin features underestimates both, since the model can lean on the twin.
* **SHAP (global and local):** splits each prediction into one contribution per feature, starting from a baseline (the average prediction). The contributions add up exactly to the predicted probability.
* **LIME (local):** for one row, creates many slightly changed copies, asks the model about them, and fits a small linear model around that row. Its weights say which feature values push this one prediction up or down.

**Not causal.** All three methods describe how *the model* uses the features to make predictions. A feature being important is not proof that it causes podium finishes: starting at the front does not by itself put a driver on the podium; a fast car and driver cause both the good grid slot and the podium, and the model only uses the grid slot as a signal of that.

#### Global: permutation importance (validation, 30 shuffles)

In [ ]:
from sklearn.inspection import permutation_importance

pr_auc_scorer = lambda clf, X, y: average_precision_score(y, clf.predict_proba(X)[:, 1])   # works for any of our models
pi = permutation_importance(FINAL_MODEL, X_val, y_val, scoring=pr_auc_scorer, n_repeats=30, random_state=SEED)
perm = pd.DataFrame({"PR-AUC drop": pi.importances_mean, "std": pi.importances_std}, index=FEATURES)

# the same idea per feature group: shuffle all columns of a group together
rng = np.random.default_rng(SEED)
full_score = pr_auc_scorer(FINAL_MODEL, X_val, y_val)
group_rows = []
for group, cols in FEATURE_GROUPS.items():
    drops = []
    for _ in range(30):
        shuffled = X_val.copy()
        shuffled[cols] = X_val[cols].iloc[rng.permutation(len(X_val))].set_axis(X_val.index)
        drops.append(full_score - pr_auc_scorer(FINAL_MODEL, shuffled, y_val))
    group_rows.append({"group": group, "PR-AUC drop": np.mean(drops), "std": np.std(drops)})
group_perm = pd.DataFrame(group_rows).set_index("group")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [3, 2]})
for ax, imp, title in [(ax1, perm, "per feature"), (ax2, group_perm, "per feature group (shuffled together)")]:
    imp = imp.sort_values("PR-AUC drop")
    ax.barh(imp.index, imp["PR-AUC drop"], xerr=imp["std"], color="tab:blue")
    ax.set_title(f"Permutation importance {title}")
    ax.set_xlabel("drop in validation PR-AUC (bar = mean, line = std)")
fig.suptitle(f"Final model: {FINAL_NAME}")
plt.tight_layout()
plt.show()
display(perm.sort_values("PR-AUC drop", ascending=False).round(4).head(8))
display(group_perm.sort_values("PR-AUC drop", ascending=False).round(4))

#### Global: SHAP summary (validation)
SHAP and LIME need numbers only, so `era` is passed as a number (its category code) and turned back into the category before the model sees it. Rows with a missing value (mostly round 1, without standings) are left out of the explanations, so every explained value is real. Each dot is one validation row. Position = how much that feature moved the predicted podium probability for that row (right = up, left = down); colour = the feature's value (red = high, blue = low).

In [ ]:
import warnings
with warnings.catch_warnings():          # shap's import prints a harmless progress-bar warning in notebooks
    warnings.simplefilter("ignore")
    import shap

CATEGORIES = {c: list(table[c].astype("category").cat.categories) for c in CATEGORICAL}   # e.g. the 3 eras

def as_numbers(X):
    """Features as plain numbers for SHAP/LIME (a category -> its code)."""
    return X.assign(**{c: pd.Categorical(X[c], categories=CATEGORIES[c]).codes for c in CATEGORICAL}).astype(float)

def predict_podium(A):
    """Podium probability for a numeric array/frame in FEATURES order (categories as codes), as SHAP and LIME need."""
    X = pd.DataFrame(np.asarray(A, dtype=float), columns=FEATURES)
    for c in CATEGORICAL:
        X[c] = pd.Categorical.from_codes(X[c].round().astype(int), categories=CATEGORIES[c])
    return FINAL_MODEL.predict_proba(X)[:, 1]

complete = val[FEATURES].notna().all(axis=1)
xai_rows = val[complete]                                   # validation rows without missing values
X_xai, y_xai = as_numbers(xai_rows[FEATURES]), xai_rows["podium"].to_numpy()
print(f"explained rows: {len(X_xai)} of {len(val)} validation rows")

background = X_xai.sample(100, random_state=SEED)          # SHAP's baseline = average prediction over these rows
explainer = shap.Explainer(predict_podium, shap.maskers.Independent(background), algorithm="permutation",
                           feature_names=FEATURES, seed=SEED)
shap_values = explainer(X_xai, max_evals=2 * len(FEATURES) + 1, silent=True)
print(f"SHAP baseline (average predicted podium probability of the background rows): {shap_values.base_values[0]:.3f}")

shap.plots.beeswarm(shap_values, max_display=12, show=False)
plt.title(f"SHAP summary of the final model ({FINAL_NAME}), validation")
plt.xlabel("SHAP value (change in predicted podium probability)")
plt.tight_layout()
plt.show()

#### Local: one podium row and one non-podium row
Chosen by a fixed rule, not by hand: the podium finisher with the **highest** predicted probability (a clear case), and the non-podium row with the **median** predicted probability among non-podiums (a typical midfield car). The finishing position and status are shown only to tell the story; they are post-race information and never features.

In [ ]:
race_names = tables["races"].set_index("raceId")["name"]
driver_names = tables["drivers"].set_index("driverId")["surname"]
finish = outcomes.set_index(["raceId", "driverId"])[["positionOrder", "status"]]

proba_xai = predict_podium(X_xai)
non_podium_rows = np.where(y_xai == 0)[0]
examples = {
    "podium": int(np.argmax(np.where(y_xai == 1, proba_xai, -1))),
    "non-podium": int(non_podium_rows[np.argsort(proba_xai[non_podium_rows])[len(non_podium_rows) // 2]]),
}

def describe(i):
    """Who/where for explained row i (finishing position and status from `outcomes`, for the story only)."""
    r = xai_rows.iloc[i]
    pos, status = finish.loc[(r["raceId"], r["driverId"])]
    return (f"{driver_names[r['driverId']]}, {race_names[r['raceId']]} {r['year']}: grid {r['grid']}, "
            f"predicted {proba_xai[i]:.1%}, finished P{pos} ({status})")

for label, i in examples.items():
    print(f"{label:>10}: {describe(i)}")

**Local SHAP (waterfall):** starts at the baseline at the bottom and adds each feature's contribution until it reaches this row's predicted probability at the top.

In [ ]:
for label, i in examples.items():
    shap.plots.waterfall(shap_values[i], max_display=10, show=False)
    plt.title(f"Local SHAP, {label} example\n{describe(i)}")
    plt.tight_layout()
    plt.show()

**LIME:** the bars are the weights of LIME's small linear model around this row (green pushes towards podium, red away from it). We also print how close LIME's local model gets to the real model's probability.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

category_cols = {FEATURES.index(c): CATEGORIES[c] for c in CATEGORICAL}
yes_no_columns = [j for j, c in enumerate(FEATURES) if c in NUMERIC and X_xai[c].nunique() <= 2]
lime_explainer = LimeTabularExplainer(X_xai.to_numpy(), feature_names=FEATURES, class_names=["no podium", "podium"],
                                      categorical_features=yes_no_columns + list(category_cols),
                                      categorical_names=category_cols, mode="classification", random_state=SEED)
predict_both = lambda A: np.column_stack([1 - predict_podium(A), predict_podium(A)])

for label, i in examples.items():
    lime_result = lime_explainer.explain_instance(X_xai.iloc[i].to_numpy(), predict_both, num_features=10)
    lime_result.as_pyplot_figure()
    plt.title(f"LIME, {label} example\n{describe(i)}")
    plt.xlabel("LIME weight (effect on the podium probability)")
    plt.tight_layout()
    plt.show()
    print(f"{label}: model probability {proba_xai[i]:.3f}, LIME's local estimate {lime_result.local_pred[0]:.3f}, "
          f"local fit R^2 {lime_result.score:.2f}")

#### The most confident miss
The non-podium row the model was most sure about: what happened in that race?

In [ ]:
most_confident_miss = int(np.argmax(np.where(y_xai == 0, proba_xai, -1)))
print(f"most confident miss: {describe(most_confident_miss)}")

**Interpretation.**
* **Qualifying and grid carry most of the signal.** Per feature, shuffling `qual_position` drops the validation PR-AUC by 0.174 and `grid` by 0.092; every other feature is below 0.03. Per group, **qualifying (0.105) and grid (0.101)** come first, then **standings (0.057)** and **form (0.027)**; **circuit history (0.005)** adds almost nothing and **context (`home_flag`, `era`) adds nothing** (-0.003).
* **Per feature vs per group:** the qualifying group drops less (0.105) than `qual_position` alone (0.174). This is expected with twin features: shuffling a whole group also shuffles the inputs that partly cancel each other inside a linear model (for example `qual_position` and `reached_q3`), while single-feature importance measures how much the model needs one column *given the others*. Neither number alone is "the" importance; that is why we show both.
* **Direction (SHAP summary):** a low (blue) `qual_position`, `grid` and `pos_before` push the probability up by up to about +0.25; a high (red) `driver_form5` and `pts_before` push it up too. Some signs look surprising, for example `reached_q3 = 1` pushes *down*: given the exact qualifying position, reaching Q3 adds nothing new, and the linear model uses it to correct the other qualifying features. This is a property of correlated inputs, not of Formula 1.
* **Podium example (Hamilton, Bahrain 2020, from pole, won):** SHAP starts at the baseline 0.16 and adds grid 1 (+0.21), his recent form (+0.16), qualifying P1 (+0.14), leading the championship (+0.09), 307 points (+0.08) and 8 earlier podiums at this circuit (+0.07), reaching 0.924. **Non-podium example (Ocon, Italy 2020, grid 12, finished 8th):** qualifying 12th (-0.06) and grid 12 (-0.05) bring him down to 0.015. Both explanations are what a fan would expect.
* **LIME agrees on the direction but is only an approximation:** it ranks `qual_position` and `grid` first, but its small linear model estimates 0.34 where the real model says 0.92 (local fit R² 0.55), and it fits the non-podium row poorly (R² 0.18). SHAP values add up exactly to the model's prediction, so we trust SHAP more for the size of each effect.
* **The most confident miss is Hamilton at the 2020 Italian Grand Prix:** pole position, predicted 90.9%, finished 7th after a stop-and-go penalty during the race. Events inside the race cannot be seen by a pre-race model; this is a limit of the task, not of the model (4.12).

**Not causal.** These explanations describe the model, not Formula 1. The model relies on the qualifying and grid positions because they are the best pre-race summary of how fast the car and driver are; it does not show that the starting position *causes* podiums, and moving a driver up the grid would not give him the model's probability.

### 4.12 Model limitations

Two checks give numbers to the main limitations.
1. **Are the predicted probabilities realistic in every split?** Every race has exactly 3 podiums, so the predicted probabilities of one race should add up to about 3. This check needs **no labels**, so it can be run on any future season before its races; we also compare the average predicted probability with the real podium rate.
2. **The false alarms on the test set:** how many of those cars did not finish the race (crash, breakdown), something no pre-race model can see?

In [ ]:
# 1) are the probabilities realistic? (podiums per race should be about 3; this part needs no labels)
calib_rows = []
for split, part in [("train", train), ("validation", val), ("test", test)]:
    p = pd.Series(FINAL_MODEL.predict_proba(part[FEATURES])[:, 1], index=part.index)
    calib_rows.append({"split": split, "predicted podiums per race (median)": p.groupby(part["raceId"]).sum().median(),
                       "average predicted probability": p.mean(), "real podium rate": part["podium"].mean()})
display(pd.DataFrame(calib_rows).set_index("split").round(3))
if CATEGORICAL and hasattr(FINAL_MODEL.named_steps["model"], "coef_"):    # the weights the categories got
    w = pd.Series(FINAL_MODEL.named_steps["model"].coef_[0], index=FINAL_MODEL.named_steps["prep"].get_feature_names_out())
    print("weights of the category columns:", w[w.index.str.startswith("cat__")].round(3).to_dict())

# 2) false alarms on the test set: how many of those cars did not finish the race?
status = test[["raceId", "driverId"]].merge(outcomes[["raceId", "driverId", "status"]],
                                            on=["raceId", "driverId"], how="left")["status"]
finished = (status.eq("Finished") | status.str.match(r"^\+\d+ Laps?$")).to_numpy()
said_podium = FINAL_MODEL.predict_proba(X_test)[:, 1] >= FINAL_THRESHOLD
false_alarm = said_podium & (y_test.to_numpy() == 0)
print(f"test false alarms: {false_alarm.sum()}; did not finish the race: {(~finished[false_alarm]).sum()} "
      f"({(~finished[false_alarm]).mean():.0%}), compared with {(~finished).mean():.0%} of all test starters")

**What we see.**
* **Probabilities:** realistic on train and validation (about 3.0 and 2.9 predicted podiums per race; average probability 0.142 vs real rates 0.142 and 0.151), but **too high on the test years: about 5.0 predicted podiums per race** (average 0.248 vs a real rate of 0.151). The weights explain it: logistic regression learned -1.28 for `hybrid` and -1.12 for `v8`; a `ground_effect` row gets neither, so its score is about 1.2 higher than the same row in a known era. The per-race check needs no labels, so it should be run on every new season before trusting the threshold.
* **False alarms:** 270 on the test set; 16% of those cars did not finish, about the same as all starters (14%). So the extra false alarms come from the shifted probabilities, not from crashes or breakdowns.

**Limitations of the models**

| Model | Main limitation in this project |
|---|---|
| Logistic regression (final) | Linear: no interactions unless they are built as features; weights of correlated features (grid, qualifying, standings) are not individually interpretable; extrapolates linearly to inputs never seen in training (the unseen era). |
| Random forest | Overfits with small leaves (train PR-AUC 0.91 vs test 0.68); probabilities are vote shares, not calibrated; cannot extrapolate beyond training values. |
| Gradient boosting | Sensitive to tree size (default settings overfit: train 0.93 vs validation 0.70); many hyperparameters for a small validation set. |
| Shallow neural network | Stopping epoch chosen on validation, so its validation score is optimistic (0.752 on validation, 0.698 on test); many settings; a black box without XAI. |

**Limitations of the data and the method**
* **Era sensitivity:** rules, car design and the points system change between eras (25 points for a win since 2010, 10 before), so the same feature value does not mean the same thing in every season. The test era (`ground_effect`) never occurs in training, and one-hot encoding turns an unseen era into an input the model has never seen (above).
* **Qualifying coverage:** reliable qualifying data starts in the mid-1990s and the knockout format in 2006, so P3's table starts in 2003; `qual_format`, `reached_q2` and `reached_q3` describe the format changes.
* **Missing values carry meaning:** in the training years a missing qualifying time almost always means a driver who set no time and started near the back, so the model reads "missing" as a bad sign (4.13: a pole-sitter without qualifying data gets about 1%). For a record with a data gap the prediction is unreliable.
* **Raw lap time:** `best_qual_s` depends on the circuit length, so its value is only comparable within one race (`gap_to_pole_pct` is the comparable version).
* **Small validation set:** 117 podiums; the bootstrap standard error of PR-AUC is about 0.04, larger than every difference between the tuned models, so the ranking of the models is uncertain.
* **The threshold is chosen for F1**, which counts a missed podium and a false alarm as equally bad. Another use (for example a betting or broadcast decision) weighs them differently and needs a different threshold.
* **The task itself:** incidents, penalties, safety cars, weather and strategy happen during the race and are invisible before it (Hamilton, Italy 2020, in 4.11). No pre-race model can predict them.

**Recommendation for the final table (P3, before the freeze):** remove `era` from the model inputs, or map an unseen era to the latest known one. On validation it adds nothing (context group importance -0.003, 4.11) and its test value cannot be learned. We do not apply this change in this notebook after seeing the test results, to keep the test set honest.

### 4.13 Hand-off to the inference function (P1)

P1's inference function uses the objects below. `FINAL_MODEL` is one pipeline (preprocessing + model), so it takes the raw feature columns of a record and handles missing values itself (4.2). Usage:

```python
proba = FINAL_MODEL.predict_proba(record[FEATURES])[:, 1]   # podium probability
podium = proba >= FINAL_THRESHOLD                           # yes / no decision
```

The check below runs the final model on one complete validation record and on the same record with its qualifying and standings values removed (as for a driver without a qualifying time in the first race of a season).

In [ ]:
complete_record = X_val[X_val.notna().all(axis=1)].iloc[[0]]
incomplete_record = complete_record.copy()
incomplete_record[["qual_position", "best_qual_s", "gap_to_pole_pct", "pts_before", "pos_before",
                   "cons_pts_before", "cons_pos_before"]] = np.nan
check = pd.concat([complete_record, incomplete_record]).assign(record=["complete", "incomplete"]).set_index("record")
check["podium probability"] = FINAL_MODEL.predict_proba(check[FEATURES])[:, 1].round(3)
check["podium (threshold)"] = check["podium probability"] >= FINAL_THRESHOLD
assert check["podium probability"].notna().all()
display(check[["grid", "qual_position", "pts_before", "podium probability", "podium (threshold)"]])
print(f"FINAL_NAME = {FINAL_NAME!r}, FINAL_THRESHOLD = {FINAL_THRESHOLD}, {len(FEATURES)} features")